Note: Written calculations may slightly differ from notebook answers since I calculated them using a calculator instead of using Python.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns 

from scipy.stats import norm

df = pd.read_csv('./data/ames_prices.csv')
df.head()

,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


Question 7

In [2]:
df['central_air_ind'] = (df['Central.Air'] == 'Y').astype(int)

In [3]:
#Part A
p_hat = df['central_air_ind'].mean() 
n_obs = len(df) 
print( 
    f"Estimated proportion of homes with central air (p_hat):" 
    f" {p_hat:.4f} ({p_hat:.2%})" ) 
print(f"Total sample size (n): {n_obs}")

Estimated proportion of homes with central air (p_hat): 0.9331 (93.31%)
Total sample size (n): 2930


In [4]:
#Part B
seed = 2026 
replications = 2500 
rng = np.random.default_rng(seed)

In [5]:
bootstrap_samples = rng.choice( 
    df['central_air_ind'].to_numpy(), 
    size=(replications, n_obs), 
    replace=True, 
    ) 
bootstrap_estimates = bootstrap_samples.mean(axis=1)
print( f"Generated {replications:,} bootstrap estimates using random seed" 
      f" {seed}." 
    )

Generated 2,500 bootstrap estimates using random seed 2026.


In [6]:
#Part C
se_boot = np.std(bootstrap_estimates, ddof=1) 
print(f"Bootstrap Standard Error (SE_boot with ddof=1): {se_boot:.5f}" )

Bootstrap Standard Error (SE_boot with ddof=1): 0.00450


In [7]:
#Part D
z_crit = {0.90: 1.645, 0.95: 1.960, 0.99: 2.576}
ci_results = {}
print("Normal Confidence Intervals:")
for level, z in z_crit.items():
    margin = z * se_boot
    ci_low = p_hat - margin
    ci_high = p_hat + margin
    ci_results[level] = (ci_low, ci_high)
    print(f" {int(level*100)}% Confidence Interval (z = {z}): [{ci_low:.4f}," f" {ci_high:.4f}]")

Normal Confidence Intervals:
 90% Confidence Interval (z = 1.645): [0.9257, 0.9405]
 95% Confidence Interval (z = 1.96): [0.9243, 0.9419]
 99% Confidence Interval (z = 2.576): [0.9215, 0.9447]


In [8]:
#Part E
p_0 = 0.90
for level, alpha in [(0.90, 0.10), (0.95, 0.05), (0.99, 0.01)]:
    ci_low, ci_high = ci_results[level]
    inside_ci = ci_low <= p_0 <= ci_high
    decision = ( "Fail to reject H0" if inside_ci else "REJECT H0" )
    print( f" At alpha = {alpha:.2f} ({int(level*100)}% CI): {decision} (0.90" f" is outside [{ci_low:.4f}, {ci_high:.4f}])" )

 At alpha = 0.10 (90% CI): REJECT H0 (0.90 is outside [0.9257, 0.9405])
 At alpha = 0.05 (95% CI): REJECT H0 (0.90 is outside [0.9243, 0.9419])
 At alpha = 0.01 (99% CI): REJECT H0 (0.90 is outside [0.9215, 0.9447])


Question 8

In [9]:
x = np.arcsinh(df['area']).to_numpy()
y = np.arcsinh(df['price']).to_numpy()
n_obs = len(df)

In [10]:
c_x = x - np.mean(x) 
c_y = y - np.mean(y) 
r_hat = np.dot(c_x, c_y) / (np.linalg.norm(c_x) * np.linalg.norm(c_y))

In [11]:
print(f"Original Point Estimate (Pearson r): {r_hat:.4f}")
print(f"Sample Size (N): {n_obs:,} observations")

Original Point Estimate (Pearson r): 0.7232
Sample Size (N): 2,930 observations


In [12]:
#Part A
boot_corrs = []
for i in range(replications):
    idx = rng.choice(n_obs, size=n_obs, replace=True)
    c_xb = x[idx] - np.mean(x[idx])
    c_yb = y[idx] - np.mean(y[idx])
    r_b = np.dot(c_xb, c_yb) / (np.linalg.norm(c_xb) * np.linalg.norm(c_yb))
    boot_corrs.append(r_b)

In [13]:
boot_corrs = np.array(boot_corrs) 
print( f"Generated {replications:,} bootstrap estimates using seed {seed}." )

Generated 2,500 bootstrap estimates using seed 2026.


In [14]:
#Part B 
#use ddof=1 for the 1/(N-1) dof correction 
se_boot = np.std(boot_corrs, ddof=1) 
print( f"Bootstrap Standard Error (SE_boot with ddof=1): {se_boot:.5f}" )

Bootstrap Standard Error (SE_boot with ddof=1): 0.00987


In [15]:
#Part C
ci_results = {}
for level, z in z_crit.items():
    margin = z * se_boot
    ci_low = r_hat - margin
    ci_high = r_hat + margin
    ci_results[level] = (ci_low, ci_high)
    print( f"{int(level*100)}% Confidence Interval (z = {z}): [{ci_low:.4f}," f" {ci_high:.4f}]" )

90% Confidence Interval (z = 1.645): [0.7069, 0.7394]
95% Confidence Interval (z = 1.96): [0.7038, 0.7425]
99% Confidence Interval (z = 2.576): [0.6977, 0.7486]


In [16]:
#Part D
rho_0 = 0.0
for level, alpha in [(0.90, 0.10), (0.95, 0.05), (0.99, 0.01)]:
    ci_low, ci_high = ci_results[level]
    inside_ci = ci_low <= rho_0 <= ci_high
    decision = "Fail to reject H0" if inside_ci else "REJECT H0"
    print( f" At alpha = {alpha:.2f} ({int(level*100)}% CI): {decision} (0.0 is" f" outside [{ci_low:.4f}, {ci_high:.4f}])" )

 At alpha = 0.10 (90% CI): REJECT H0 (0.0 is outside [0.7069, 0.7394])
 At alpha = 0.05 (95% CI): REJECT H0 (0.0 is outside [0.7038, 0.7425])
 At alpha = 0.01 (99% CI): REJECT H0 (0.0 is outside [0.6977, 0.7486])


Part E written on paper.

Question 9

In [17]:
df = pd.read_csv('./data/metabric.csv')

In [18]:
survival_months = df['Overall Survival (Months)'].dropna().to_numpy()
n_obs = len(survival_months)

In [19]:
mean_obs_months = np.mean(survival_months)
mean_obs_years = mean_obs_months / 12.0

In [20]:
print(f"Sample Size (N): {n_obs:,} patients" ) 
print(f"Sample Mean Survival: {mean_obs_months:.2f} months ({mean_obs_years:.2f} years)\n")

Sample Size (N): 1,343 patients
Sample Mean Survival: 129.09 months (10.76 years)



In [21]:
#Part A
seed = 2026 
replications = 2500 
rng = np.random.default_rng(seed)

In [22]:
boot_samples = rng.choice(survival_months, size=(replications, n_obs), replace=True) 
boot_means = boot_samples.mean(axis=1)

In [23]:
print(f"Random-number seed: {seed}")
print(f"Number of bootstrap replications (R): {replications:,}\n")

Random-number seed: 2026
Number of bootstrap replications (R): 2,500



In [24]:
se_boot_months = np.std(boot_means, ddof=1) 
se_boot_years = se_boot_months / 12.0

In [25]:
#Part B
print(f"Bootstrap Standard Error (SE_boot): {se_boot_months:.4f} months ({se_boot_years:.4f} years)\n")

Bootstrap Standard Error (SE_boot): 2.1424 months (0.1785 years)



In [26]:
#Part C
ci_results = {} 
for level, z in z_crit.items(): 
    margin_m = z * se_boot_months 
    ci_low_m, ci_high_m = mean_obs_months - margin_m, mean_obs_months + margin_m 
    ci_low_y, ci_high_y = ci_low_m / 12.0, ci_high_m / 12.0 
    ci_results[level] = (ci_low_m, ci_high_m) 
    print(f"{int(level*100)}% Confidence Interval (z = {z}):") 
    print(f" Months: [{ci_low_m:.2f}, {ci_high_m:.2f}]") 
    print(f" Years: [{ci_low_y:.2f}, {ci_high_y:.2f}]")

90% Confidence Interval (z = 1.645):
 Months: [125.57, 132.62]
 Years: [10.46, 11.05]
95% Confidence Interval (z = 1.96):
 Months: [124.90, 133.29]
 Years: [10.41, 11.11]
99% Confidence Interval (z = 2.576):
 Months: [123.58, 134.61]
 Years: [10.30, 11.22]


In [27]:
#Part D
h_0 = 120.0
for level, alpha in [(0.90, 0.10), (0.95, 0.05), (0.99, 0.01)]:
    ci_low_m, ci_high_m = ci_results[level]
    inside_ci = ci_low_m <= h_0 <= ci_high_m
    decision = "Fail to reject H0" if inside_ci else "REJECT H0"
    print(f"At alpha = {alpha:.2f} ({int(level*100)}% CI): {decision}") 
    print(f"120.0 is {'INSIDE' if inside_ci else 'OUTSIDE'} [{ci_low_m:.2f}, {ci_high_m:.2f}]")

At alpha = 0.10 (90% CI): REJECT H0
120.0 is OUTSIDE [125.57, 132.62]
At alpha = 0.05 (95% CI): REJECT H0
120.0 is OUTSIDE [124.90, 133.29]
At alpha = 0.01 (99% CI): REJECT H0
120.0 is OUTSIDE [123.58, 134.61]


Part E written on paper.

Question 10

In [28]:
df = pd.read_csv('./data/nhanes_data_17_18.csv', low_memory=False)

In [29]:
age_mask = df['AgeInYearsAtScreening'] >= 18
sleep_mask = df['SleepHoursWeekdaysOrWorkdays'].notna()

In [30]:
df_adults = df[age_mask & sleep_mask].copy()
sleep_data = df_adults['SleepHoursWeekdaysOrWorkdays'].to_numpy()
n_obs = len(sleep_data)
mean_sleep_obs = np.mean(sleep_data)

In [31]:
print(f"N: {n_obs:,}")       
print(f"Sample Mean Weekday Sleep (X_bar): {mean_sleep_obs:.4f} hours\n")

N: 5,489
Sample Mean Weekday Sleep (X_bar): 7.6212 hours



In [32]:
#Part A
boot_samples = rng.choice(sleep_data, size=(replications, n_obs), replace=True)
boot_means = boot_samples.mean(axis=1)

In [33]:
print(f"Random number seed: {seed}") 
print(f"Number of bootstrap replications (R): {replications:,}\n")

Random number seed: 2026
Number of bootstrap replications (R): 2,500



In [34]:
#Part B 
se_boot = np.std(boot_means, ddof=1)

In [35]:
print(f"Bootstrap Standard Error: {se_boot:.5f} hours\n" )

Bootstrap Standard Error: 0.02266 hours



In [36]:
#Part C
ci_results = {} 
for level, z in z_crit.items(): 
    margin = z * se_boot 
    ci_low = mean_sleep_obs - margin 
    ci_high = mean_sleep_obs + margin 
    ci_results[level] = (ci_low, ci_high) 
    print( f"{int(level*100)}% Confidence Interval (z = {z}): [{ci_low:.4f}," f" {ci_high:.4f}] hours" )

90% Confidence Interval (z = 1.645): [7.5840, 7.6585] hours
95% Confidence Interval (z = 1.96): [7.5768, 7.6657] hours
99% Confidence Interval (z = 2.576): [7.5629, 7.6796] hours


In [37]:
#Part D
h_0 = 8.0 
for level, alpha in [(0.90, 0.10), (0.95, 0.05), (0.99, 0.01)]: 
    ci_low, ci_high = ci_results[level] 
    inside_ci = ci_low <= h_0 <= ci_high 
    decision = "Fail to reject H0" if inside_ci else "REJECT H0" 
    print( f"At alpha = {alpha:.2f} ({int(level*100)}% CI): {decision} (8.0" f" hours is {'INSIDE' if inside_ci else 'OUTSIDE'} [{ci_low:.4f}," f" {ci_high:.4f}])" )

At alpha = 0.10 (90% CI): REJECT H0 (8.0 hours is OUTSIDE [7.5840, 7.6585])
At alpha = 0.05 (95% CI): REJECT H0 (8.0 hours is OUTSIDE [7.5768, 7.6657])
At alpha = 0.01 (99% CI): REJECT H0 (8.0 hours is OUTSIDE [7.5629, 7.6796])


Part E written on paper.